# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sorgerator/flyrank-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### How I Built the Queue

A raw machine learning prediction like `probability = 0.78` tells me that a page is likely declining, but it doesn't tell an editor **what to actually do** with that page. If I just hand an editorial team a list of probabilities, they won't know whether to rewrite the whole article, tweak the title tag, or fix on-page engagement.

To make this practical and trustworthy, I combined my **Week 5 Random Forest model** with my **Week 4 baseline rules** into a prioritized action queue:

1. **Model Probability ($P(\text{decline})$)**: Trained on an honest, client-grouped split (`client_id` holdout) using search and engagement features, strictly excluding leakage columns (`trend_direction`, `trend_pct`, `is_declining_label`).
2. **Baseline Heuristic Score**: A normalized score combining historical visibility (log-impressions), freshness risk (days since last update), rank opportunity (positions 1–20), and content depth gaps (<1,200 words).
3. **Blended Score**: $100 \times (0.70 \times P(\text{decline}) + 0.30 \times \text{baseline score})$, giving us a smooth 0–100 priority score.
4. **Reason Codes**: Diagnostic tags that tell the human reviewer *why* the page was flagged (e.g. `HIGH_RANK_LOW_CTR`, `THIN_VISIBLE_PAGE`, `LOW_ENGAGEMENT_VISIBLE_PAGE`, `MODEL_DECLINE_RISK`).
5. **Suggested Action**: A specific recommended task mapped from the reason codes (e.g. `UPDATE_META_TAGS`, `EXPAND_AND_REFRESH`, `REFRESH_AND_REVIEW_CTR`, `REFRESH`, `MONITOR`).
6. **Confidence Gating**: I label items `HIGH`, `MEDIUM`, or `LOW` confidence so editors only prioritize pages that have enough data (at least 500 impressions and 10 sessions) rather than low-traffic noise.

In [1]:
import os
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# 1. Load Data (supporting local and Colab paths)
if os.path.exists('../../data/raw/content_refresh_anonymized.csv'):
    data_path = '../../data/raw/content_refresh_anonymized.csv'
    output_dir = Path('../../work/outputs')
elif os.path.exists('data/raw/content_refresh_anonymized.csv'):
    data_path = 'data/raw/content_refresh_anonymized.csv'
    output_dir = Path('work/outputs')
else:
    data_path = 'content_refresh_anonymized.csv'
    output_dir = Path('outputs')

output_dir.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(data_path)

# 2. Recreate the honest target and feature set from Week 5 & 6
df['is_declining_label'] = (df['trend_direction'].str.lower() == 'down').astype(int)
target_col = 'is_declining_label'
group_col = 'client_id'
leakage_cols = ['content_id', 'client_id', 'trend_direction', 'trend_pct', target_col]

numeric_cols = df.select_dtypes(include=[np.number, bool]).columns
feature_cols = [c for c in numeric_cols if c not in leakage_cols]

# 3. Fit Random Forest on the Honest Grouped Split (by client_id)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df[group_col]))

rf_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
    ('model', RandomForestClassifier(random_state=42, n_estimators=100, max_depth=5))
])

rf_pipeline.fit(df.iloc[train_idx][feature_cols], df.iloc[train_idx][target_col])
df['model_probability'] = rf_pipeline.predict_proba(df[feature_cols])[:, 1]

# 4. Calculate Normalized Heuristic Baseline Score
def min_max_normalize(series: pd.Series) -> pd.Series:
    vals = pd.to_numeric(series, errors='coerce').fillna(0)
    mi, ma = vals.min(), vals.max()
    return pd.Series(0.0, index=vals.index) if ma == mi else (vals - mi) / (ma - mi)

df['visibility_score'] = min_max_normalize(np.log1p(df['impressions_90d'].clip(lower=0)))
df['freshness_risk_score'] = min_max_normalize(df['days_since_last_update'])
df['position_opp_score'] = np.where(
    (df['avg_position'] > 0) & (df['avg_position'] <= 20),
    1.0 - (df['avg_position'] / 20.0),
    0.0
)
df['depth_gap_score'] = np.where((df['word_count'] > 0) & (df['word_count'] < 1200), 1.0, 0.0)

df['baseline_refresh_score'] = (
    0.40 * df['visibility_score'] +
    0.30 * df['freshness_risk_score'] +
    0.25 * df['position_opp_score'] +
    0.05 * df['depth_gap_score']
)
df['baseline_score_normalized'] = min_max_normalize(df['baseline_refresh_score'])

# 5. Blended Final Priority Score (0 to 100)
df['final_refresh_score'] = (
    100.0 * (0.70 * df['model_probability'] + 0.30 * df['baseline_score_normalized'])
).clip(0, 100)

# 6. Assign Diagnostic Reason Codes
def get_reason_codes(row: pd.Series) -> str:
    reasons = []
    if row['model_probability'] >= 0.65:
        reasons.append('MODEL_DECLINE_RISK')
    if row['model_probability'] >= 0.50 and row['impressions_90d'] >= 500:
        reasons.append('VISIBLE_MODEL_OPPORTUNITY')
    if row['impressions_90d'] >= 500 and (0 < row['avg_position'] <= 20) and row['ctr'] < 0.5:
        reasons.append('HIGH_RANK_LOW_CTR')
    if row['sessions_90d'] >= 30 and ((0 < row['engagement_rate'] < 30) or (0 < row['scroll_rate'] < 30)):
        reasons.append('LOW_ENGAGEMENT_VISIBLE_PAGE')
    if row['days_since_last_update'] >= 180 and row['impressions_90d'] >= 500:
        reasons.append('STALE_VISIBLE_PAGE')
    if 0 < row['word_count'] < 1200 and row['impressions_90d'] >= 250:
        reasons.append('THIN_VISIBLE_PAGE')
    return '|'.join(reasons) if reasons else 'MONITOR_CANDIDATE'

df['reason_codes'] = df.apply(get_reason_codes, axis=1)

# 7. Map Reason Codes to Actionable Suggestions
def get_suggested_action(row: pd.Series) -> str:
    reasons = set(str(row['reason_codes']).split('|'))
    if 'THIN_VISIBLE_PAGE' in reasons:
        return 'EXPAND_AND_REFRESH'
    if 'HIGH_RANK_LOW_CTR' in reasons and ('MODEL_DECLINE_RISK' in reasons or 'VISIBLE_MODEL_OPPORTUNITY' in reasons):
        return 'REFRESH_AND_REVIEW_CTR'
    if 'LOW_ENGAGEMENT_VISIBLE_PAGE' in reasons and ('MODEL_DECLINE_RISK' in reasons or 'VISIBLE_MODEL_OPPORTUNITY' in reasons):
        return 'REFRESH_AND_REVIEW_ENGAGEMENT'
    if 'HIGH_RANK_LOW_CTR' in reasons:
        return 'UPDATE_META_TAGS'
    if {'MODEL_DECLINE_RISK', 'VISIBLE_MODEL_OPPORTUNITY', 'STALE_VISIBLE_PAGE'}.intersection(reasons):
        return 'REFRESH'
    return 'MONITOR'

df['suggested_action'] = df.apply(get_suggested_action, axis=1)

# 8. Evidence-Gated Confidence Tiers
p80 = float(df['final_refresh_score'].quantile(0.80))
p50 = float(df['final_refresh_score'].quantile(0.50))

def get_confidence_tier(row: pd.Series) -> str:
    if (
        row['final_refresh_score'] >= p80 and
        row['impressions_90d'] >= 500 and
        row['sessions_90d'] >= 10 and
        row['model_probability'] >= 0.50
    ):
        return 'HIGH'
    elif row['final_refresh_score'] >= p50:
        return 'MEDIUM'
    return 'LOW'

df['confidence'] = df.apply(get_confidence_tier, axis=1)

# 9. Sort and Rank the Queue
df_ranked = df.sort_values(
    by=['final_refresh_score', 'impressions_90d', 'sessions_90d'],
    ascending=[False, False, False]
).reset_index(drop=True)
df_ranked['final_rank'] = df_ranked.index + 1

# Display the Top 20 Candidates
cols_to_show = [
    'final_rank', 'content_id', 'final_refresh_score', 'model_probability',
    'suggested_action', 'reason_codes', 'confidence',
    'avg_position', 'ctr', 'impressions_90d', 'sessions_90d', 'trend_direction'
]
print(f"Total portfolio items scored: {len(df_ranked):,}")
print(f"High-confidence items with strong evidence: {(df_ranked['confidence'] == 'HIGH').sum():,}")
print(f"80th Percentile Score Cutoff: {p80:.2f} | Median Cutoff: {p50:.2f}\n")
print("Top 20 Ranked Action Queue:")
print(df_ranked[cols_to_show].head(20).to_string(index=False))


Total portfolio items scored: 30,000
High-confidence items with strong evidence: 2,916
80th Percentile Score Cutoff: 64.47 | Median Cutoff: 57.82

Top 20 Ranked Action Queue:
 final_rank           content_id  final_refresh_score  model_probability              suggested_action                                                                               reason_codes confidence  avg_position  ctr  impressions_90d  sessions_90d trend_direction
          1 content_7a6df559322d            79.851249           0.741028        REFRESH_AND_REVIEW_CTR MODEL_DECLINE_RISK|VISIBLE_MODEL_OPPORTUNITY|HIGH_RANK_LOW_CTR|LOW_ENGAGEMENT_VISIBLE_PAGE       HIGH           0.7 0.14            43650            69            down
          2 content_1dfabe7b1fe8            79.311283           0.779514        REFRESH_AND_REVIEW_CTR MODEL_DECLINE_RISK|VISIBLE_MODEL_OPPORTUNITY|HIGH_RANK_LOW_CTR|LOW_ENGAGEMENT_VISIBLE_PAGE       HIGH           2.9 0.49             9582            40            down
          3 

### Top-20 Queue Review

When I inspect the top 20 items in my queue, the logic makes clear intuitive sense:
- **Page 1 Visibility with Low CTR (`REFRESH_AND_REVIEW_CTR`)**: Items like `content_7a6df559322d` (Rank 1) and `content_1dfabe7b1fe8` (Rank 2) rank high on page 1 (positions 0.7 and 2.9) with massive impression volume (43,650 and 9,582 impressions), but have tiny CTRs (0.14% and 0.49%). The model gives them high decline probabilities (>0.74). These are high-priority candidates where updating meta titles and descriptions could yield immediate click recovery.
- **Engagement Drops (`REFRESH_AND_REVIEW_ENGAGEMENT`)**: Item `content_69fad7e6c50c` (Rank 5) gets good impressions (28,000) and sessions (345), but gets flagged for low engagement/scroll rates. Here, an editor knows not to change the SEO title, but rather to improve the on-page content and UX.
- **Why this builds trust**: An editor doesn't have to guess why a page is at the top of the list. The reason codes tell them exactly what the data saw.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [2]:
# Summary of Queue Volumes by Action and Confidence Tier
action_conf_table = pd.crosstab(
    df_ranked['suggested_action'],
    df_ranked['confidence'],
    margins=True,
    margins_name='Total'
)

action_stats = df_ranked.groupby('suggested_action').agg(
    count=('content_id', 'count'),
    avg_score=('final_refresh_score', 'mean'),
    avg_model_prob=('model_probability', 'mean'),
    avg_impressions=('impressions_90d', 'mean'),
    avg_sessions=('sessions_90d', 'mean'),
    declining_rate=('is_declining_label', 'mean')
).sort_values(by='count', ascending=False)

print("Action by Confidence Breakdown:")
print(action_conf_table.to_string())

print("\nAction Category Profile Summary:")
print(action_stats.round(3).to_string())


Action by Confidence Breakdown:
confidence                     HIGH    LOW  MEDIUM  Total
suggested_action                                         
EXPAND_AND_REFRESH               10     35      37     82
MONITOR                           0   9498     212   9710
REFRESH                         377   1343    6445   8165
REFRESH_AND_REVIEW_CTR         2156   1264    3931   7351
REFRESH_AND_REVIEW_ENGAGEMENT   373    544    1385   2302
UPDATE_META_TAGS                  0   2316      74   2390
Total                          2916  15000   12084  30000

Action Category Profile Summary:
                               count  avg_score  avg_model_prob  avg_impressions  avg_sessions  declining_rate
suggested_action                                                                                              
MONITOR                         9710     33.455           0.339         3509.047        35.995           0.262
REFRESH                         8165     61.411           0.714         1082.46

### 1. Who uses this and for what?
- **Who**: SEO specialists, content managers, and editorial teams managing large websites.
- **For what**: As a **decision-support tool** to triage weekly content audits. Instead of manually checking 30,000 pages or guessing, editors start at the top of the **2,916 High-Confidence items** to spend their limited time where the data shows clear decay and high traffic potential.

### 2. Where does this stop being valid? (Honest Limits)
Following the Claim Ladder rules (`skills/writing-honest-claims/SKILL.md`), I must be honest about what this model can and cannot do:

1. **Decision-support, not causal proof**: My model found patterns *associated* with traffic decline. It **does not prove** that editing a page will cause Google to rank it higher. Traffic recovery depends on competitor content and editorial quality.
2. **Cold-start limitation**: The model is invalid for new content (<90 days old) or pages with 0 impressions. Without historical traffic data, these pages need manual editorial review, not decay scoring.
3. **External and site-wide factors**: If a client did a full website redesign or got hit with a domain-wide penalty, all their pages will look "declining" even if the individual articles are fine. Seasonality (like winter coat guides in summer) can also look like decay.
4. **No algorithm claims**: I did not "crack Google's ranking algorithm." I built an honest machine learning ranking system that achieved a measured Precision@50 of 98.0% (compared to the 51.1% base rate and 34.0% baseline rule) on unseen client portfolios.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [3]:
# Identifying Edge Cases That Need Extra Human Checking
# Conflict Case: High predicted decline probability (>0.70) but currently ranking in top 3 spots
high_rank_conflicts = df_ranked[
    (df_ranked['model_probability'] >= 0.70) &
    (df_ranked['avg_position'] > 0) &
    (df_ranked['avg_position'] <= 3.0) &
    (df_ranked['impressions_90d'] >= 500)
]

print(f"Top-3 Ranking Pages with High Decline Probability: {len(high_rank_conflicts):,} pages")
if len(high_rank_conflicts) > 0:
    print("\nSample items where human review is required before touching:")
    print(high_rank_conflicts[[
        'final_rank', 'content_id', 'final_refresh_score', 'model_probability',
        'avg_position', 'ctr', 'impressions_90d', 'sessions_90d', 'suggested_action'
    ]].head(5).to_string(index=False))


Top-3 Ranking Pages with High Decline Probability: 178 pages

Sample items where human review is required before touching:
 final_rank           content_id  final_refresh_score  model_probability  avg_position  ctr  impressions_90d  sessions_90d       suggested_action
          1 content_7a6df559322d            79.851249           0.741028           0.7 0.14            43650            69 REFRESH_AND_REVIEW_CTR
          2 content_1dfabe7b1fe8            79.311283           0.779514           2.9 0.49             9582            40 REFRESH_AND_REVIEW_CTR
          3 content_6e098546a7a2            78.186606           0.754739           1.9 0.13            10575            12 REFRESH_AND_REVIEW_CTR
          6 content_66458ac1b739            76.733734           0.751448           2.9 0.03             6822             3 REFRESH_AND_REVIEW_CTR
          7 content_87f1ffe0bedb            76.480421           0.733814           1.7 0.23             8225            13 REFRESH_AND_REVIEW_CTR


### 1. What a human must check before taking action

Before editing or updating any flagged page, an editor must manually verify:
1. **SERP Layout & AI Overviews**: Did clicks drop because Google added an AI Overview, Featured Snippet, or Shopping block at the top? If so, the CTR drop is due to Google's interface, not bad content.
2. **Competitor Brands & Query Intent**: Is the page accidentally ranking for a competitor's brand name? People searching for a specific brand won't click our article, which naturally results in a low CTR.
3. **Internal Cannibalization**: Did another newer article on our own site take over the traffic? If so, we should canonicalize or merge rather than rewrite.
4. **Technical Errors**: Is the page broken, redirecting, or missing tracking tags? A technical bug should be fixed by developers, not content writers.

---

### 2. The No-Go List (What must NEVER be automated)

- **Never auto-delete or auto-redirect pages**: Removing content automatically can destroy valuable backlinks and break user navigation.
- **Never publish AI-generated rewrites without human editing**: LLMs can hallucinate and write off-brand text. An editor must always review drafts before publishing.
- **Never auto-update core transactional or legal pages**: Checkout pages, pricing tables, and terms of service must always be handled manually.
- **Never automate changes during active Google Core Updates**: Search rankings fluctuate wildly during core updates. Automated changes should be frozen until the search landscape settles.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [4]:
# Telemetry Benchmarks for Monitoring Drift
telemetry_stats = {
    'total_portfolio_items': int(len(df_ranked)),
    'target_decline_base_rate': float(round(df_ranked['is_declining_label'].mean(), 4)),
    'model_precision_at_50_benchmark': 0.980,
    'baseline_rule_precision_at_50': 0.340,
    'high_confidence_ratio': float(round((df_ranked['confidence'] == 'HIGH').mean(), 4)),
    'impression_quantiles': {
        'p25': float(df_ranked['impressions_90d'].quantile(0.25)),
        'p50': float(df_ranked['impressions_90d'].quantile(0.50)),
        'p75': float(df_ranked['impressions_90d'].quantile(0.75)),
        'p90': float(df_ranked['impressions_90d'].quantile(0.90))
    },
    'ctr_by_position_bracket': {
        str(k): float(round(v, 4)) for k, v in df_ranked[df_ranked['avg_position'] > 0].groupby(
            pd.cut(df_ranked['avg_position'], bins=[0, 3, 10, 20, 50, 100]),
            observed=False
        )['ctr'].mean().to_dict().items()
    }
}

print("Baseline Telemetry Benchmarks:")
print(json.dumps(telemetry_stats, indent=2))


Baseline Telemetry Benchmarks:
{
  "total_portfolio_items": 30000,
  "target_decline_base_rate": 0.5421,
  "model_precision_at_50_benchmark": 0.98,
  "baseline_rule_precision_at_50": 0.34,
  "high_confidence_ratio": 0.0972,
  "impression_quantiles": {
    "p25": 81.0,
    "p50": 731.0,
    "p75": 3615.25,
    "p90": 12136.400000000009
  },
  "ctr_by_position_bracket": {
    "(0, 3]": 2.7143,
    "(3, 10]": 0.651,
    "(10, 20]": 0.3234,
    "(20, 50]": 0.2223,
    "(50, 100]": 0.1525
  }
}


### How I Would Know Recommendations Went Stale

A content playbook goes stale when search behavior or site performance shifts. I would monitor three key areas:

1. **Input Data Drift (Covariate Shift)**: If average CTR across top ranking positions drops significantly (e.g. from 2.7% down to 1.2% in positions 1–3), it tells me Google changed the search layout. The model's baseline assumptions need updating.
2. **Model Performance Decay (Concept Drift)**: If my model's Precision@50 on new client holdout data drops below 60% (getting close to the 51.1% random guess rate), the model is no longer picking good refresh targets.
3. **Feedback Loop Problems**: If we refresh the top 50 pages and they recover, retraining the model on those same pages later without noting they were edited will confuse the model.

### My Retraining Triggers

- **Calendar Trigger**: Retrain every 90 days with the freshest trailing-90-day data window.
- **Accuracy Trigger**: Retrain immediately if holdout Precision@50 drops below 60%.
- **Human Feedback Trigger**: Retrain or adjust rules if editors reject more than 30% of the top recommended actions.
- **Search Update Trigger**: Retrain after any major confirmed Google Core Algorithm Update once the 14-day volatility window passes.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [5]:
# 1. Export Full Ranked Action Queue
queue_file = output_dir / 'action_playbook_queue.csv'
queue_cols = [
    'final_rank', 'content_id', 'client_id', 'final_refresh_score',
    'model_probability', 'baseline_refresh_score', 'confidence',
    'suggested_action', 'reason_codes', 'is_declining_label',
    'impressions_90d', 'clicks_90d', 'sessions_90d', 'avg_position',
    'ctr', 'content_age_days', 'days_since_last_update', 'word_count',
    'trend_direction'
]
df_ranked[queue_cols].to_csv(queue_file, index=False)
print(f"Saved queue to: {queue_file} ({len(df_ranked):,} rows)")

# 2. Export Visualizations for the Research Paper
charts_dir = output_dir / 'charts'
charts_dir.mkdir(parents=True, exist_ok=True)

# Chart 1: Action Mix
plt.figure(figsize=(8, 4.5))
action_counts = df_ranked['suggested_action'].value_counts()
action_counts.plot(kind='barh', color='#426B69')
plt.title('Suggested Actions Distribution across Portfolio', fontsize=12)
plt.xlabel('Number of Content Items', fontsize=10)
plt.gca().invert_yaxis()
plt.tight_layout()
chart1_path = charts_dir / 'action_mix.png'
plt.savefig(chart1_path, dpi=200)
plt.close()

# Chart 2: Confidence Tier Breakdown
plt.figure(figsize=(6, 4))
conf_counts = df_ranked['confidence'].value_counts().reindex(['HIGH', 'MEDIUM', 'LOW'], fill_value=0)
conf_counts.plot(kind='bar', color=['#2CA02C', '#FF7F0E', '#7F7F7F'])
plt.title('Action Queue Confidence Tiers', fontsize=12)
plt.ylabel('Number of Pages', fontsize=10)
plt.xticks(rotation=0)
plt.tight_layout()
chart2_path = charts_dir / 'confidence_mix.png'
plt.savefig(chart2_path, dpi=200)
plt.close()

# Chart 3: Top Reason Codes
reasons_count = {}
for r_text in df_ranked['reason_codes']:
    for code in str(r_text).split('|'):
        reasons_count[code] = reasons_count.get(code, 0) + 1
top_reasons = pd.Series(reasons_count).sort_values(ascending=False).head(7)

plt.figure(figsize=(8, 4.5))
top_reasons.plot(kind='barh', color='#4E79A7')
plt.title('Top Diagnostic Reason Codes in Queue', fontsize=12)
plt.xlabel('Count of Pages Triggering Reason', fontsize=10)
plt.gca().invert_yaxis()
plt.tight_layout()
chart3_path = charts_dir / 'top_reason_codes.png'
plt.savefig(chart3_path, dpi=200)
plt.close()

# 3. Export Summary JSON for Paper Ingestion
summary_dict = {
    'assignment': 'ML-10',
    'total_rows_scored': int(len(df_ranked)),
    'high_confidence_rows': int((df_ranked['confidence'] == 'HIGH').sum()),
    'medium_confidence_rows': int((df_ranked['confidence'] == 'MEDIUM').sum()),
    'low_confidence_rows': int((df_ranked['confidence'] == 'LOW').sum()),
    'action_breakdown': df_ranked['suggested_action'].value_counts().to_dict(),
    'top_reason_codes': top_reasons.to_dict(),
    'model_precision_at_50': 0.980,
    'baseline_precision_at_50': 0.340,
    'queue_file': str(queue_file),
    'chart_files': [str(chart1_path), str(chart2_path), str(chart3_path)]
}

summary_file = output_dir / 'action_playbook_summary.json'
with open(summary_file, 'w') as f:
    json.dump(summary_dict, f, indent=2)

print(f"Saved summary JSON to: {summary_file}")
print(f"Saved charts to: {charts_dir}")


Saved queue to: ..\..\work\outputs\action_playbook_queue.csv (30,000 rows)
Saved summary JSON to: ..\..\work\outputs\action_playbook_summary.json
Saved charts to: ..\..\work\outputs\charts


### Exported Files for the Capstone Research Paper (ML-11)

I have exported the following files to `work/outputs/` to use in my final research paper:

| Exported File | What It Contains | Where It Goes in the Paper |
|---|---|---|
| `work/outputs/action_playbook_queue.csv` | Full ranked queue with scores, actions, and reason codes | Section 6: Recommendations |
| `work/outputs/charts/action_mix.png` | Bar chart of suggested action breakdown | Section 7: Embedded Charts |
| `work/outputs/charts/confidence_mix.png` | Bar chart of High/Medium/Low confidence volume | Section 6 & 7: Risk & Priorities |
| `work/outputs/charts/top_reason_codes.png` | Bar chart of top diagnostic reason codes | Section 6: Interpretability |
| `work/outputs/action_playbook_summary.json` | JSON summary of key metrics and counts | Appendix & Build verification |

## Self-check

Before you submit, confirm each line honestly:

- [**X**] Every section above is filled — markdown thinking AND the code that backs it
- [**X**] The notebook runs top to bottom with no errors (Runtime → Run all)
- [**X**] No client names, URLs, or private queries anywhere
- [**X**] My claims use careful words: observed, measured, directional, decision-support
- [**X**] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.